In [13]:
import pandas as pd
import numpy as np

In [14]:
raw_data = pd.read_parquet("raw_yfinance.parquet")

In [15]:
tickers = {
    "Gold": "GC=F",          # Gold futures
    "Brent": "BZ=F",         # Brent crude futures
    "DXY": "DX-Y.NYB",       # US Dollar Index
    "VIX": "^VIX",           # CBOE Volatility Index
    "US10Y": "^TNX",         # 10Y Treasury yield, quoted as yield x 10
}

In [16]:
# Use Adj Close if available, otherwise Close
if "Adj Close" in raw_data.columns.get_level_values(0):
    prices = raw_data["Adj Close"].copy()
else:
    prices = raw_data["Close"].copy()

# Rename ticker columns to readable asset names
ticker_to_name = {ticker: name for name, ticker in tickers.items()}
prices = prices.rename(columns=ticker_to_name)

# Sort by date
prices = prices.sort_index()

# Remove rows where everything is missing
prices = prices.dropna(how="all")

prices.head()

Ticker,Brent,DXY,Gold,US10Y,VIX
Date,,,,,
2010-01-04,80.120003,77.529999,1117.699951,3.841,20.040001
2010-01-05,80.589996,77.620003,1118.099976,3.755,19.350000
2010-01-06,81.889999,77.489998,1135.900024,3.808,19.160000
2010-01-07,81.510002,77.910004,1133.099976,3.822,19.059999
2010-01-08,81.370003,77.470001,1138.199951,3.808,18.129999


In [17]:
print("Date range:")
print("Start:", prices.index.min())
print("End:", prices.index.max())
print("")
print("Shape:", prices.shape)
print("")
print("Missing values:")
print(prices.isna().sum())

prices.tail()

Date range:
Start: 2010-01-04 00:00:00
End: 2026-05-29 00:00:00

Shape: (4132, 5)

Missing values:
Ticker
Brent    37
DXY       5
Gold      7
US10Y     8
VIX       5
dtype: int64


Ticker,Brent,DXY,Gold,US10Y,VIX
Date,,,,,
2026-05-25,NaN,NaN,NaN,NaN,16.590000
2026-05-26,99.580002,99.169998,4500.399902,4.493,17.010000
2026-05-27,94.290001,99.209999,4447.500000,4.481,16.290001
2026-05-28,93.709999,99.019997,4499.299805,4.455,15.740000
2026-05-29,91.120003,98.910004,4593.000000,4.453,15.320000


In [18]:
# Forward-fill small holiday mismatches
prices_clean = prices.ffill()

# Drop rows where any required variable is still missing
core_assets = ["Gold", "Brent", "DXY", "VIX", "US10Y"]
prices_clean = prices_clean[core_assets].dropna()

prices_clean.head()

Ticker,Gold,Brent,DXY,VIX,US10Y
Date,,,,,
2010-01-04,1117.699951,80.120003,77.529999,20.040001,3.841
2010-01-05,1118.099976,80.589996,77.620003,19.350000,3.755
2010-01-06,1135.900024,81.889999,77.489998,19.160000,3.808
2010-01-07,1133.099976,81.510002,77.910004,19.059999,3.822
2010-01-08,1138.199951,81.370003,77.470001,18.129999,3.808


In [19]:
print("Date range:")
print("Start:", prices_clean.index.min())
print("End:", prices_clean.index.max())
print("")
print("Shape:", prices_clean.shape)
print("")
print("Missing values:")
print(prices_clean.isna().sum())

prices_clean.tail()

Date range:
Start: 2010-01-04 00:00:00
End: 2026-05-29 00:00:00

Shape: (4132, 5)

Missing values:
Ticker
Gold     0
Brent    0
DXY      0
VIX      0
US10Y    0
dtype: int64


Ticker,Gold,Brent,DXY,VIX,US10Y
Date,,,,,
2026-05-25,4521.000000,103.540001,99.320000,16.590000,4.558
2026-05-26,4500.399902,99.580002,99.169998,17.010000,4.493
2026-05-27,4447.500000,94.290001,99.209999,16.290001,4.481
2026-05-28,4499.299805,93.709999,99.019997,15.740000,4.455
2026-05-29,4593.000000,91.120003,98.910004,15.320000,4.453


In [20]:
prices_clean.to_parquet("clean_yfinance.parquet")